# Notizen für Quantize

# Gyro

Zahlen, die quantisiert werden sollen:
- score(): Alle außer var355, Probas[] und Output[]
- infer(): Alles außer result


# Imports

In [189]:
import re
import pandas as pd

# Find largest float in C File

In [190]:
def find_largest_float_in_c_file(filename, retMin = False):
    try:
        with open(filename, 'r') as file:
            content = file.read()
        
        # Regex für Gleitkommazahlen: +/- Zahlen mit Dezimalpunkt (kann führende und nachfolgende Ziffern oder nicht haben)
        pattern = re.compile(r'[-+]?\d*\.\d+(?:[eE][-+]?\d+)?') # 
        
        # Finde alle Gleitkommazahlen
        floats = pattern.findall(content)
        
        # Konvertiere alle gefundenen Gleitkommazahlen in tatsächliche Floats
        float_numbers = [float(num) for num in floats]
        
        # Gibt die größte Zahl zurück, falls die Liste nicht leer ist
        if float_numbers:
            if retMin == False: return max(float_numbers)
            if retMin == True: return min(float_numbers)
        else:
            return None

    except FileNotFoundError:
        print(f"Die Datei {filename} wurde nicht gefunden.")
        return None
    except Exception as e:
        print(f"Ein unerwarteter Fehler ist aufgetreten: {e}")
        return None


# Replace Floats in C File

Multiplikation mit 100.000.000, weil im Modell die größte Gleitkommazahl 
13.* und die kleinste Zahl -1.* ist.\
Datentyp Long kann [-2.147.483.648;2.147.483.647] abbilden \
2147483647 / 13,353132 = 160822468,241908 -> stark gerundet 100000000

In [191]:
def replace_floats_in_c_file(filename):
    try:
        with open(filename, 'r') as file:
            content = file.read()
        
        # Regex für Gleitkommazahlen
        float_pattern = re.compile(r'[-+]?\d*\.\d+(?:[eE][-+]?\d+)?')

        # Funktion, um gefundene Gleitkommazahlen zu ersetzen
        def replace_function(match):
            float_str = match.group(0)
            float_number = float(float_str)
            modified_number = round(float_number * 100000000)
            return str(modified_number)

        # Ersetze alle Gleitkommazahlen im Inhalt
        modified_content = float_pattern.sub(replace_function, content)

        modified_content = re.sub(r'\bdouble\b', 'long', modified_content, flags=re.IGNORECASE)


        # Schreibe die Änderungen zurück in die Datei oder in eine neue Datei
        with open(filename, 'w') as file:
            file.write(modified_content)
    
        print(f"Alle Gleitkommazahlen in '{filename}' wurden erfolgreich ersetzt.")

    except FileNotFoundError:
        print(f"Die Datei {filename} wurde nicht gefunden.")
    except Exception as e:
        print(f"Ein unerwarteter Fehler ist aufgetreten: {e}")


# Main


### Modify C-Files

In [ ]:
# filename = 'gyro_model_ino_quantized.c'
# filename = 'infer_quantized.c'

# max_float = find_largest_float_in_c_file(filename)
# min_float = find_largest_float_in_c_file(filename, retMin = True)
# print(f"Die größte Gleitkommazahl in der Datei ist: {max_float}")
# print(f"Die kleinste Gleitkommazahl in der Datei ist: {min_float}")

# replace_floats_in_c_file(filename)

### Create (1st) Compare-CSV

In [ ]:
# bc = pd.read_csv('baseCapture.csv', sep=';')
# nqf = pd.read_csv('no_quantized_float.csv', sep=';')
# nqd = pd.read_csv('no_quantized_double.csv', sep=';')

# df = pd.DataFrame({
#     "baseScore0": bc['baseScore_0'],
#     "inoScore0_float": nqf['inoScore0'],
#     "inoScore0_double": nqd['inoScore0'],
#     "baseScore1": bc['baseScore_1'],
#     "inoScore1_float": nqf['inoScore1'],
#     "inoScore1_double": nqd['inoScore1'],
#     "label": bc['label'],
#     "inoLabel_float": nqf['inoLabel'],
#     "inoLabel_double": nqd['inoLabel']
#     }
# )

# df = df.iloc[:250]

# df['inoLabel_float'] = df['inoLabel_float'].astype(int)
# df['inoLabel_double'] = df['inoLabel_double'].astype(int)

# df.to_csv('compare.csv', sep=',', index=False)